# 10 — La libreria in un giro: C#
Stessa sequenza in **C#** e in **PySharp**, stessi dati (stesso seme) e stessi numeri: ogni riga `etichetta: valori` stampata qui compare identica nell'altro notebook, e il test `TourNotebooksAgreeAcrossLanguages` lo verifica. In C# si usa l'API **generica** (`T = double`); in PySharp la facciata `MathemaSharp.TimeSeries.Interop`, perché la sua reflection non passa `ReadOnlySpan<T>` né risolve i metodi generici.

Kernel **JupyterNet** (C#) su runtime **.NET 11** (`scripts/run-notebooks.ps1`). Parte II del libro: [la libreria](../../refs/timeseries/01-libreria/README.md).

In [ ]:
#load "../shared/MathemaSharp.csx"
using MathemaSharp.TimeSeries;
using System.Globalization;

static void P(string label, IEnumerable<double> values) =>
    Console.WriteLine($"{label}: {string.Join(" ", values.Select(v => v.ToString("F6", CultureInfo.InvariantCulture)))}");
static void P(string label, double value) => P(label, new[] { value });

// Segnale simulato: AR(2) con φ = (0.6, −0.2), σ = 1, 600 campioni. Ultimi 20 = test.
var y = Simulation.Arma<double>(new Random(2026), 600, 0, new[] { 0.6, -0.2 }, Array.Empty<double>(), 1);
double[] train = y[..580], test = y[580..];
P("n", y.Length);

## 1 · Statistica di base (cap. 2)

In [ ]:
P("mean", SeriesStatistics.Mean<double>(train));
P("acf", SeriesStatistics.Acf<double>(train, 5));
P("pacf", SeriesStatistics.Pacf<double>(train, 5));
var lb = SeriesStatistics.LjungBox<double>(train, 10, 0);
P("ljungbox", new[] { lb.Statistic, lb.PValue });
P("quantili", new[] { SeriesStatistics.Quantile<double>(train, 0.1), SeriesStatistics.Quantile<double>(train, 0.5), SeriesStatistics.Quantile<double>(train, 0.9) });
Chart.Stems("PACF: si taglia dopo il lag 2", SeriesStatistics.Pacf<double>(train, 12), 1.96 / Math.Sqrt(train.Length));

## 2 · Riferimenti e metriche (cap. 3)

In [ ]:
foreach (var (name, f) in new (string, double[])[] {
    ("mean", ReferenceForecasts.Mean<double>(train, 20)),
    ("naive", ReferenceForecasts.Naive<double>(train, 20)),
    ("drift", ReferenceForecasts.Drift<double>(train, 20)) })
    P($"mase_{name}", Accuracy.Mase<double>(test, f, train, 1));

## 3 · AR e ARIMA (cap. 4)

In [ ]:
var ar = ArModel.FitYuleWalker<double>(train, 2);
P("ar_phi", ar.Phi);
P("ar_sigma2", ar.InnovationVariance);
P("ar_forecast", ar.Forecast(train, 5));
var arma = ArimaModel.FitHannanRissanen<double>(train, 1, 0, 1);
P("arma_params", new[] { arma.Constant, arma.Phi[0], arma.Theta[0], arma.InnovationVariance });

## 4 · Smoothing esponenziale (cap. 5)

In [ ]:
var alpha = ExponentialSmoothing.FitSesAlpha<double>(train);
P("ses_alpha", alpha);
P("ses_forecast", ExponentialSmoothing.Ses<double>(train, alpha, 3).Forecast);
P("holt_damped", ExponentialSmoothing.Holt<double>(train, 0.4, 0.2, 5, 0.9).Forecast);

// Serie con trend e stagionalità annuale (periodo 12), sullo stesso rumore.
var seasonal = y.Select((v, t) => v + 0.02 * t + 3 * Math.Sin(2 * Math.PI * t / 12)).ToArray();
var hw = ExponentialSmoothing.HoltWinters<double>(seasonal[..588], 12, 0.3, 0.05, 0.2, 12, 1.0);
P("hw_forecast", hw.Forecast[..6]);
P("hw_mase", Accuracy.Mase<double>(seasonal[588..], hw.Forecast, seasonal[..588], 12));

## 5 · Filtro di Kalman (cap. 6)

In [ ]:
var kf = KalmanFilter.LocalLevel<double>(0.1, 1.0, train[0], 10.0);
var kr = kf.Filter(train[..100]);
P("kalman_gain_last", kr.FirstGain[^1]);
P("kalman_loglik", kr.LogLikelihood);
P("kalman_steady_gain", KalmanFilter.LocalLevelSteadyStateGain<double>(0.1, 1.0));

## 6 · VAR (cap. 7)
Due serie accoppiate: $x$ = i primi 400 campioni, $z_t=0.5\,z_{t-1}+0.3\,x_{t-1}+0.5\,e_t$.

In [ ]:
var noise = Enumerable.Range(0, 400).Select(i => Simulation.StandardNormal(new Random(7 + i))).ToArray();
var data = new double[400, 2];
for (var t = 0; t < 400; t++)
{
    data[t, 0] = y[t];
    data[t, 1] = t == 0 ? 0.5 * noise[t] : 0.5 * data[t - 1, 1] + 0.3 * y[t - 1] + 0.5 * noise[t];
}
var var1 = VarModel.Fit<double>(data, 1);
P("var_a1", new[] { var1.Coefficients[0][0, 0], var1.Coefficients[0][0, 1], var1.Coefficients[0][1, 0], var1.Coefficients[0][1, 1] });
P("var_stable", var1.IsStableOrder1Bivariate() ? 1 : 0);

## 7 · GARCH (cap. 8)

In [ ]:
var truth = new Garch11<double>(0.05, 0.1, 0.85);
var shocks = Simulation.Garch(new Random(7), truth, 1500);
var fit = Garch11.FitVarianceTargeting<double>(shocks);
P("garch_fit", new[] { fit.Omega, fit.Alpha, fit.Beta });
P("garch_forecast", fit.ForecastVariance(shocks, 5));

## 8 · STL (cap. 9)

In [ ]:
var stl = Stl.Decompose<double>(seasonal, 12, 7, null, 2, 0);
P("stl_trend_ends", new[] { stl.Trend[0], stl.Trend[^1] });
P("stl_seasonal", stl.Seasonal[..12]);
Chart.Panels("STL della serie con trend e stagionalità", new Line("osservato", seasonal), new Line("trend", stl.Trend), new Line("stagionalità", stl.Seasonal), new Line("resto", stl.Remainder));

## 9 · Solo in C#: lo stesso algoritmo in altri formati
Levinson-Durbin sulla stessa serie in `float`, `double` e `Decimal128`. La differenza fra i risultati è solo aritmetica (stesso rumore): la PACF al lag 1 cambia dalla settima cifra in `float` alla sedicesima in `double`.

In [ ]:
using System.Numerics;
foreach (var (name, value) in new (string, double)[]
{
    ("float", double.CreateChecked(SeriesStatistics.Pacf<float>(train.Select(float.CreateChecked).ToArray(), 1)[0])),
    ("double", SeriesStatistics.Pacf<double>(train, 1)[0]),
    ("decimal128", double.CreateChecked(SeriesStatistics.Pacf<Decimal128>(train.Select(Decimal128.CreateChecked).ToArray(), 1)[0])),
})
    Console.WriteLine($"pacf1_{name}: {value:R}");